# 05 Basic Stacking

Train the logistic regression stacker using validation predictions from the three base models.

In [ ]:
import pandas as pd

from src.arimax_model import train_arimax_model
from src.data_loader import ProjectConfig, load_price_data
from src.feature_engineering import engineer_features, get_feature_sets, split_frame
from src.lstm_model import train_lstm_model
from src.stacking import train_logistic_stacker
from src.xgboost_model import train_xgboost_model

config = ProjectConfig()
feature_df = engineer_features(load_price_data(config), config)
train_df, validation_df, test_df = split_frame(feature_df)
feature_sets = get_feature_sets()

xgb = train_xgboost_model(train_df, validation_df, test_df, feature_sets['xgboost'])
arimax = train_arimax_model(train_df, validation_df, test_df, feature_sets['arimax_exog'])
lstm = train_lstm_model(train_df, validation_df, test_df, feature_sets['lstm'], config.lstm_window)

validation_base = pd.DataFrame({
    'xgb_prob': xgb['validation_probabilities'],
    'lstm_prob': lstm['validation_probabilities'],
    'arimax_pred': arimax['validation_probabilities'],
}, index=validation_df.index)
test_base = pd.DataFrame({
    'xgb_prob': xgb['test_probabilities'],
    'lstm_prob': lstm['test_probabilities'],
    'arimax_pred': arimax['test_probabilities'],
}, index=test_df.index)

logistic_results = train_logistic_stacker(validation_base, validation_df['target'], test_base, test_df['target'])
logistic_results['validation_metrics']

In [ ]:
logistic_results['test_metrics']